# Lab 02 · Retrieve architecture rules with RAG

**Prepared by Mustafa Assaf** · 15–20 minute lab

## Retrieve the policy instead of pasting it

**So far:** in Lab 01 we pasted the policy into every prompt.

**New:** keep several policies in a small store, *retrieve* the ones that match the question, and give only that evidence to the model. This is retrieval-augmented generation (RAG). We also check that the answer cites the evidence it was given, and we skip the model when nothing matches.

Our retriever counts the words that a question and a policy share. Real systems often use embeddings and a vector database, but the steps are the same: score, rank, keep the strong matches, then answer from them.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, save `FREEINFERENCE_API_KEY` in Colab Secrets and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/aebb52597ab982f0b8b2f0832815b73dd6aca416.zip"
from coursekit import make_ask
MODE = "practice"
ask = make_ask(mode=MODE)

## Step 1 · Store a few policies

Each policy has an id and a text. Only R1 is about our architecture question. The others are there so that retrieval has something to reject.

In [ ]:
policies = {
    "R1": "Choose microservices only when the team has 8+ people and services must deploy independently; "
          "otherwise choose a modular monolith.",
    "R2": "Each module owns its tables. One module never reads the tables of another module.",
    "R3": "Deploy to the eu-west region unless a customer contract names another region.",
    "R4": "Public APIs keep the old version running for 6 months after a breaking change.",
}
print(len(policies), "policies:", list(policies))

## Step 2 · Score each policy against a question

`words` turns text into a *set* of lower-case words, without punctuation and without *stop words* such as "the" and "should", which appear everywhere. Then `&` keeps only the words that two sets share. `score` counts the shared words for every policy.

In [ ]:
STOP_WORDS = {"a", "an", "the", "of", "or", "and", "to", "we", "our", "should", "can", "what", "which", "on", "is"}

def words(text):
    """The lower-case words in a text, without punctuation or stop words."""
    return {word.strip(".,;:?!") for word in text.lower().split()} - STOP_WORDS

def score(question):
    """How many words each policy shares with the question."""
    return {pid: len(words(question) & words(text)) for pid, text in policies.items()}

question = "Our team of 3 people ships one release. Should we choose microservices or a modular monolith?"
print("Shared with R1:", sorted(words(question) & words(policies["R1"])))
print("Scores:", score(question))

## Step 3 · Keep only the strong matches

`retrieve` sorts the policy ids from the highest score to the lowest and keeps the top two. It then drops any policy that shares fewer than `min_shared` words. A single shared word, such as "team", is too weak to count as evidence. The lunch question shows why.

In [ ]:
def retrieve(question, min_shared=2, top_k=2):
    """The ids of the best-matching policies, best first. Weak matches are dropped."""
    scores = score(question)
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [pid for pid in ranked[:top_k] if scores[pid] >= min_shared]

questions = [
    "Our team of 3 people ships one release. Should we choose microservices or a modular monolith?",
    "Can the billing module read the tables of the orders module?",
    "What should the team eat for lunch on Friday?",
]
for q in questions:
    print(q)
    print("    scores:", score(q), "-> retrieved:", retrieve(q))

## Step 4 · Build a grounded prompt

The prompt holds only the retrieved policies, each labelled with its id, and it asks the model to cite an id like `[R1]`. Policies that were not retrieved never reach the model.

In [ ]:
def grounded_prompt(question, ids):
    """A prompt that holds only the retrieved policies, each labelled with its id."""
    evidence = "\n".join(f"[{pid}] {policies[pid]}" for pid in ids)
    return (f"Answer using only these policies:\n{evidence}\n\n"
            f"Question: {question}\nAnswer in one sentence and cite the policy id, like [R1].")

print(grounded_prompt(questions[0], retrieve(questions[0])))

## Step 5 · Answer, abstain, and check the citations

Our code runs two checks around the model call:

- When nothing was retrieved, the model is **not called**. The program abstains instead of letting the model guess.
- `re.findall(r"R\d+", answer)` finds the cited ids. An answer is *grounded* when it cites at least one id and every cited id was retrieved. In Python, `cited <= set(ids)` asks "is `cited` a subset of `ids`?".

In [ ]:
import re

scripted = ["A modular monolith: 3 people is below the 8-person threshold [R1].",
            "No. Each module owns its tables, so billing must ask the orders module [R2].",
            ""]  # the lunch question never reaches the model
verdicts = []  # one True/False per answered question
for question, example in zip(questions, scripted):
    ids = retrieve(question)
    if not ids:
        print(f"Q: {question}\nA: No matching policy, so the model is not called.\n")
        continue
    answer = ask(grounded_prompt(question, ids), example=example)
    cited = set(re.findall(r"R\d+", answer))
    grounded = bool(cited) and cited <= set(ids)
    verdicts.append(grounded)
    print(f"Q: {question}\nA: {answer}\nCited {sorted(cited)}, retrieved {ids}, grounded: {grounded}\n")
assert all(verdicts), "Each answer should cite only the policies it was given."

## What you should see

Step 2 scores R1 at 6 and the others at 0 or 1. In Step 3, the architecture question retrieves `['R1']`, the billing question retrieves `['R2']`, and the lunch question retrieves `[]`. The lunch question shares one word with R1 ("team") and one with R4 ("for"), which is below the threshold. Step 4 prints a prompt that holds only R1. In Step 5 both answers are `grounded: True`, and the lunch question is never sent to the model.

## Try it

1. Add the question `"Which region should we deploy to?"` to `questions`, and a scripted reply that cites `[R3]` to `scripted`. Predict its scores first.
2. Set `min_shared=7` in `retrieve`. Now even the architecture question finds nothing. A threshold that is too strict makes the system abstain when it should answer.
3. Change the first scripted reply to cite `[R4]`. R4 was never retrieved, so the answer is not grounded and the final check fails.

## Takeaway

RAG adds evidence that the model did not memorise, and it lets your code check the answer against that evidence. The retriever decides what the model sees, so test the retriever as carefully as the prompt. Lab 03 goes further: instead of reading text, the program calls *tools*.

## Read further

- [The original RAG paper](https://arxiv.org/abs/2005.11401)